In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, JSONLoader, CSVLoader, WebBaseLoader, ImageCaptionLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from glob import glob

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
for p in glob( r'.\test_source\*.pdf'):
    print( p )

In [3]:
alldoc = []
alldoc.extend( ["Document1", "Document2"] )
alldoc.extend( ["Document3", "Document4"] )
alldoc

['Document1', 'Document2', 'Document3', 'Document4']

In [6]:
alldoc = []
for p in glob( r'..\test_source\*.pdf'):
    print( p )
    pdfloader = PyPDFLoader(p)
    document = pdfloader.load()

    txt_split = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=20)
    docs = txt_split.split_documents( document)
    alldoc.extend( docs)

..\test_source\「FIS 이슈 & 포커스」 22-4호 《중앙-지방 간 재정조정제도》.pdf
..\test_source\「FIS 이슈 & 포커스」 23-2호 《핵심재정사업 성과관리》.pdf
..\test_source\「FIS 이슈 & 포커스」(신규) 통권 제1호 《우발부채》.pdf
..\test_source\「FIS 이슈&포커스」 22-2호 《재정성과관리제도》.pdf
..\test_source\국토교통부_행복주택출자.pdf
..\test_source\보건복지부_노인장기요양보험 사업운영.pdf
..\test_source\보건복지부_부모급여(영아수당) 지원.pdf
..\test_source\산업통상자원부_에너지바우처.pdf
..\test_source\중소벤처기업부_혁신창업사업화자금(융자).pdf


In [7]:
embedding = OpenAIEmbeddings( model='text-embedding-ada-002')
vdb = FAISS.from_documents( alldoc, embedding)
#vdb.add_documents

In [8]:
# 6. 프롬프트 템플릿 정의
template = """질문에 대해 아래의 제공된 문맥(context)만을 사용하여 답변하세요. 
답을 모른다면 모른다고 말하고 답변을 지어내지 마세요. 
최대한 세 문장 내로 간결하게 답변하세요.

Question: {question} 
Context: {context} 
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI( model='gpt-4o-mini', temperature=0)
retriever = vdb.as_retriever()  # vdb.as_retriever(k=4)

def format_docs(docs):
    #print( "문서:",docs.page_content)
    for d in docs:
        print( "페이지번호:", d.metadata['page_label'], end=" " )
        print('페이지소스:', d.metadata['source'] )
        print("====================")
    return "\n\n".join( doc.page_content for doc in docs )

qa = ( 
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt 
        | llm 
        | StrOutputParser() 
    )
query = "2022년 혁신창업사업화자금(융자)의 예산은 얼마인가요?"
result = qa.invoke( query)
print( result )

페이지번호: 5 페이지소스: ..\test_source\「FIS 이슈 & 포커스」 23-2호 《핵심재정사업 성과관리》.pdf
페이지번호: 2 페이지소스: ..\test_source\중소벤처기업부_혁신창업사업화자금(융자).pdf
페이지번호: 1 페이지소스: ..\test_source\중소벤처기업부_혁신창업사업화자금(융자).pdf
페이지번호: 8 페이지소스: ..\test_source\「FIS 이슈 & 포커스」 23-2호 《핵심재정사업 성과관리》.pdf
2022년 혁신창업사업화자금(융자)의 예산은 2,300억원입니다.
